# Nuclear NPV simulation

Run the nuclear electricity Monte Carlo simulation and visualize the resulting NPV distribution.

The summary also reports how many simulations have non-negative NPV (NPV >= 0) and how many have negative NPV.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from electricity.electricity_npv_monte_carlo import (
    DEFAULT_RANDOM_SEED,
    DEFAULT_SAMPLE_SIZE,
    simulate_electricity_results,
)

from npv_summary import summarize_metric_signs


In [2]:
TECHNOLOGY = 'nuclear'
SAMPLE_SIZE = DEFAULT_SAMPLE_SIZE
RANDOM_SEED = DEFAULT_RANDOM_SEED

results_by_technology = simulate_electricity_results(
    sample_size=SAMPLE_SIZE,
    random_seed=RANDOM_SEED,
    technologies=(TECHNOLOGY,),
)
simulation = results_by_technology[TECHNOLOGY]
results = pd.DataFrame(simulation)
results.head()


,run_id,technology,annual_output_mwh,full_load_hours_per_year,lifetime_years,capacity_mw,capacity_kw,capex_eur_per_kw,fixed_opex_eur_per_kw_year,variable_opex_eur_per_mwh,...,annual_variable_opex_eur,annual_fuel_cost_eur,annual_emissions_cost_eur,annual_total_cost_eur,annual_net_cash_flow_eur,npv_eur,discounted_lifetime_output_mwh,present_value_total_cost_eur,lcoe_eur_per_mwh,levelized_net_margin_eur_per_mwh
0,0,nuclear,1000000.0,7900.0,45.0,126.582278,126582.278481,15724.587499,102.953740,6.987695,...,6.987695e+06,7.899256e+06,0.0,2.791907e+07,6.615093e+07,-1.189472e+09,1.210840e+07,2.328509e+09,192.305270,-98.235270
1,1,nuclear,1000000.0,7900.0,45.0,126.582278,126582.278481,15964.531102,97.379906,6.372797,...,6.372797e+06,8.366024e+06,0.0,2.706539e+07,6.700461e+07,-1.209508e+09,1.210840e+07,2.348545e+09,193.959983,-99.889983
2,2,nuclear,1000000.0,7900.0,45.0,126.582278,126582.278481,7427.021501,101.561229,6.895060,...,6.895060e+06,7.844292e+06,0.0,2.759520e+07,6.647480e+07,-1.352258e+08,1.210840e+07,1.274263e+09,105.237930,-11.167930
3,3,nuclear,1000000.0,7900.0,45.0,126.582278,126582.278481,6065.722358,97.025903,8.171383,...,8.171383e+06,7.743660e+06,0.0,2.819680e+07,6.587320e+07,2.980617e+07,1.210840e+07,1.109231e+09,91.608390,2.461610
4,4,nuclear,1000000.0,7900.0,45.0,126.582278,126582.278481,9114.025618,116.206434,7.746482,...,7.746482e+06,8.099498e+06,0.0,3.055566e+07,6.351434e+07,-3.846169e+08,1.210840e+07,1.523654e+09,125.834469,-31.764469


In [3]:
npv_million_eur = results["npv_eur"] / 1_000_000
lcoe_eur_per_mwh = results["lcoe_eur_per_mwh"]
levelized_net_margin_eur_per_mwh = results["levelized_net_margin_eur_per_mwh"]
summary = pd.concat(
    [
        npv_million_eur.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "NPV million EUR"
        ),
        lcoe_eur_per_mwh.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "LCOE EUR/MWh"
        ),
        levelized_net_margin_eur_per_mwh.describe(percentiles=[0.05, 0.5, 0.95]).rename(
            "Levelized net margin EUR/MWh"
        ),
    ],
    axis=1,
)

npv_signs = summarize_metric_signs(npv_million_eur)
npv_sign_summary = pd.DataFrame(
    {
        "NPV category": ["Non-negative (NPV >= 0)", "Negative (NPV < 0)"],
        "Simulation count": [
            npv_signs["non_negative_count"],
            npv_signs["negative_count"],
        ],
        "Simulation share": [
            npv_signs["non_negative_share"],
            1.0 - npv_signs["non_negative_share"],
        ],
    }
)

display(summary)
display(npv_sign_summary)


,NPV million EUR,LCOE EUR/MWh,Levelized net margin EUR/MWh
count,100000.000000,100000.000000,100000.000000
mean,-597.043875,143.378232,-49.308232
std,365.086682,30.151518,30.151518
min,-1275.443333,87.755325,-105.335401
5%,-1166.204485,96.413359,-96.313662
50%,-596.112378,143.301303,-49.231303
95%,-28.374327,190.383662,-2.343359
max,76.460623,199.405401,6.314675


,NPV category,Simulation count,Simulation share
0,Non-negative (NPV >= 0),2873,0.02873
1,Negative (NPV < 0),97127,0.97127


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(npv_million_eur, bins=50, color="tab:gray", edgecolor="white", alpha=0.8)
ax.axvline(npv_million_eur.mean(), color="tab:blue", linewidth=2, label="Mean")
ax.axvline(npv_million_eur.median(), color="tab:orange", linewidth=2, label="Median")
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"Nuclear NPV distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("NPV (million EUR)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()

/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42945/856664418.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## levelized net margin histogram


In [5]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    levelized_net_margin_eur_per_mwh,
    bins=50,
    color="tab:green",
    edgecolor="white",
    alpha=0.8,
)
ax.axvline(
    levelized_net_margin_eur_per_mwh.mean(),
    color="tab:blue",
    linewidth=2,
    label="Mean",
)
ax.axvline(
    levelized_net_margin_eur_per_mwh.median(),
    color="tab:orange",
    linewidth=2,
    label="Median",
)
ax.axvline(0, color="black", linewidth=1, linestyle="--", label="Break-even")

ax.set_title(f"Nuclear levelized net margin distribution (n={SAMPLE_SIZE})")
ax.set_xlabel("Levelized net margin (EUR/MWh)")
ax.set_ylabel("Frequency")
ax.grid(axis="y", alpha=0.25)
ax.legend()

fig.tight_layout()
plt.show()


/var/folders/0_/fgqpbggj0m725hd_1s29xmwm0000gn/T/ipykernel_42945/2693394150.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
annual_components = results[
    [
        "annual_revenue_eur",
        "annual_fixed_opex_eur",
        "annual_variable_opex_eur",
        "annual_fuel_cost_eur",
        "annual_emissions_cost_eur",
        "annual_net_cash_flow_eur",
    ]
] / 1_000_000

annual_components.mean().rename("Mean annual value, million EUR")

annual_revenue_eur           94.070000
annual_fixed_opex_eur        13.080129
annual_variable_opex_eur      7.230404
annual_fuel_cost_eur          8.008347
annual_emissions_cost_eur     0.000000
annual_net_cash_flow_eur     65.751120
Name: Mean annual value, million EUR, dtype: float64